# 99 — Consensus Delta-ML (Multi-Base-Model)

**Motivation:** nb76 uses a single LGBM to predict the delta. But LGBM can overfit to noise in sparse similarity regions. Using 3 diverse base models for delta prediction and averaging reduces variance.

**Strategy:**
1. Build the same delta-pair dataset as nb76
2. Train 3 delta models: LGBM, XGBoost (DART), Ridge regression
3. For each query compound, predict delta from each model and average (equal or OOF-optimized weights)
4. Combine with direct LGBM prediction via blending
5. Save OOF/test arrays for inclusion in grand ensemble v9

**Why Ridge?** Compressed FP features are high-dimensional; Ridge is regularized differently from tree ensembles and captures linear combinations that trees may miss.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"): sys.stdout.reconfigure(encoding="utf-8")
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, standardize_smiles, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)
from sklearn.linear_model import RidgeCV
import xgboost as xgb


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R2={r2:.4f} "
              f"r={pr:.4f} rho={sp:.4f} tau={kt:.4f}{ca}")
    return m


In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
if len(cliff_pairs) > 0:
    s2i = {s:i for i,s in enumerate(tr["smiles"].tolist())}
    ac = "cliff_active_smiles" if "cliff_active_smiles" in cliff_pairs.columns else "smiles_a"
    ic = "cliff_inactive_smiles" if "cliff_inactive_smiles" in cliff_pairs.columns else "smiles_b"
    cliff_pairs["idx_active"]   = cliff_pairs[ac].map(s2i)
    cliff_pairs["idx_inactive"] = cliff_pairs[ic].map(s2i)
    cliff_pairs = cliff_pairs.dropna(subset=["idx_active","idx_inactive"])
    cliff_pairs[["idx_active","idx_inactive"]] = cliff_pairs[["idx_active","idx_inactive"]].astype(int)
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")


Train 4,139  Test 513  Cliffs 0


In [4]:
# ---- Build delta-pair training dataset ----
SIM_LO = 0.35
SIM_HI = 0.90
MAX_PAIRS = 350_000
props = ["mw","logp","tpsa","hbd","hba","rotbonds","rings"]

print("Computing physchem descriptors...", flush=True)
phys_tr_list = tr["smiles"].map(compute_physchem).tolist()
phys_arr = np.array([[p.get(k,0) or 0 for k in props] for p in phys_tr_list], dtype=np.float32)

print("Computing pairwise Tanimoto (train x train)...", flush=True)
dot_tt = (fps_tr @ fps_tr.T).astype(np.float32)
rowsum = fps_tr.sum(1).astype(np.float32)
union_tt = rowsum[:,None] + rowsum[None,:] - dot_tt
tanimoto_tr = np.where(union_tt>0, dot_tt/union_tt, 0.0)
np.fill_diagonal(tanimoto_tr, 0.0)

i_idx, j_idx = np.where((tanimoto_tr >= SIM_LO) & (tanimoto_tr <= SIM_HI))
mask_upper = i_idx < j_idx
i_idx, j_idx = i_idx[mask_upper], j_idx[mask_upper]
print(f"Pairs in sim window [{SIM_LO},{SIM_HI}]: {len(i_idx):,}")

rng = np.random.default_rng(SEED)
if len(i_idx) > MAX_PAIRS:
    sel = rng.choice(len(i_idx), MAX_PAIRS, replace=False)
    i_idx, j_idx = i_idx[sel], j_idx[sel]
    print(f"Downsampled to {MAX_PAIRS:,} pairs")


Computing physchem descriptors...


Computing pairwise Tanimoto (train x train)...


Pairs in sim window [0.35,0.9]: 5,177


In [5]:
# ---- Feature engineering for delta prediction ----

def compress_fp(fp, out_dim=64):
    N, D = fp.shape
    block = D // out_dim
    return fp[:, :block*out_dim].reshape(N, out_dim, block).mean(-1).astype(np.float32)

def make_delta_feats(fp_anchor, fp_query, sim_col, anchor_pec50, phys_diff):
    fp_common = np.minimum(fp_anchor, fp_query).astype(np.float32)
    fp_diff   = np.abs(fp_anchor - fp_query).astype(np.float32)
    c64 = compress_fp(fp_common)
    d64 = compress_fp(fp_diff)
    return np.hstack([c64, d64, sim_col, anchor_pec50[:,None], phys_diff])

fps_i = fps_tr[i_idx]; fps_j = fps_tr[j_idx]
sims_ij = tanimoto_tr[i_idx, j_idx][:,None]
phys_diff_ij = phys_arr[j_idx] - phys_arr[i_idx]

print("Building feature matrix...", flush=True)
F_ij = make_delta_feats(fps_i, fps_j, sims_ij, y_tr[i_idx], phys_diff_ij)
F_ji = make_delta_feats(fps_j, fps_i, sims_ij, y_tr[j_idx], -phys_diff_ij)
F_all = np.vstack([F_ij, F_ji])
y_all = np.concatenate([y_tr[j_idx]-y_tr[i_idx], y_tr[i_idx]-y_tr[j_idx]])
print(f"Delta dataset: {F_all.shape}  delta range [{y_all.min():.2f}, {y_all.max():.2f}]")


Building feature matrix...


Delta dataset: (10354, 137)  delta range [-4.68, 4.68]


In [6]:
# ---- Train 3 delta base models ----
from xgboost import XGBRegressor
from sklearn.linear_model import RidgeCV

print("Training LGBM delta model...", flush=True)
DELTA_LGBM = dict(n_estimators=600, num_leaves=63, learning_rate=0.05,
                  min_child_samples=20, subsample=0.8, colsample_bytree=0.7,
                  reg_alpha=0.05, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)
delta_lgbm = lgb.LGBMRegressor(**DELTA_LGBM)
delta_lgbm.fit(F_all, y_all, callbacks=[lgb.log_evaluation(-1)])
print("LGBM delta model trained.", flush=True)

print("Training XGBoost DART delta model...", flush=True)
delta_xgb = XGBRegressor(
    n_estimators=500, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.7,
    booster="dart", rate_drop=0.1,
    tree_method="hist", random_state=SEED, verbosity=0, n_jobs=4
)
delta_xgb.fit(F_all, y_all)
print("XGBoost DART delta model trained.", flush=True)

print("Training Ridge delta model...", flush=True)
delta_ridge = RidgeCV(alphas=[0.01,0.1,1.0,10.0,100.0], cv=5)
delta_ridge.fit(F_all, y_all)
print(f"Ridge delta model trained. Best alpha={delta_ridge.alpha_:.4f}", flush=True)


Training LGBM delta model...


LGBM delta model trained.


Training XGBoost DART delta model...


XGBoost DART delta model trained.


Training Ridge delta model...


Ridge delta model trained. Best alpha=0.1000


In [7]:
# ---- Single-template delta prediction (nearest neighbor only) ----
def predict_single_template_delta(fps_query, fps_ref, y_ref, phys_query, phys_ref,
                                   delta_models, direct_preds, sim_lo=SIM_LO):
    """
    For each query, find nearest template >= sim_lo.
    Predict delta from each model, average, add to template pEC50.
    Falls back to direct_preds if no template found.
    Returns: (consensus_delta_pred, best_sims)
    """
    dot = (fps_query @ fps_ref.T).astype(np.float32)
    rs_q = fps_query.sum(1)[:,None]; rs_r = fps_ref.sum(1)[None,:]
    sim_mat = dot / np.maximum(rs_q + rs_r - dot, 1e-6)

    N = len(fps_query)
    preds = np.full(N, np.nan)
    best_sims = sim_mat.max(1)

    for qi in range(N):
        sim_row = sim_mat[qi]
        best_ri = sim_row.argmax()
        best_sim = sim_row[best_ri]
        if best_sim < sim_lo:
            preds[qi] = direct_preds[qi]
            continue
        fp_q = fps_query[qi:qi+1]
        fp_r = fps_ref[best_ri:best_ri+1]
        phys_d = phys_query[qi:qi+1] - phys_ref[best_ri:best_ri+1]
        F_q = make_delta_feats(fp_r, fp_q, np.array([[best_sim]]),
                               y_ref[best_ri:best_ri+1], phys_d)
        delta_preds_k = [m.predict(F_q)[0] for m in delta_models]
        avg_delta = float(np.mean(delta_preds_k))
        preds[qi] = float(y_ref[best_ri]) + avg_delta

    return preds, best_sims

print("Single-template consensus function ready.")


Single-template consensus function ready.


In [8]:
# ---- Scaffold 5-fold CV ----
print("\n=== Scaffold 5-fold CV ===", flush=True)
oof_consensus = np.full(len(y_tr), np.nan)
oof_direct    = np.full(len(y_tr), np.nan)
oof_delta_lgbm  = np.full(len(y_tr), np.nan)

for fold, (tr_idx, va_idx) in enumerate(splits):
    # Direct LGBM
    m_dir = lgb.train(LGBM, lgb.Dataset(X_tr[tr_idx], label=y_tr[tr_idx]),
                      valid_sets=[lgb.Dataset(X_tr[va_idx], label=y_tr[va_idx])],
                      callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof_direct[va_idx] = m_dir.predict(X_tr[va_idx])

    fps_va = fps_tr[va_idx]; fps_ft = fps_tr[tr_idx]
    phys_va = phys_arr[va_idx]; phys_ft = phys_arr[tr_idx]
    y_ft = y_tr[tr_idx]

    # LGBM-only delta (for comparison)
    preds_lgbm, sims_lgbm = predict_single_template_delta(
        fps_va, fps_ft, y_ft, phys_va, phys_ft,
        [delta_lgbm], oof_direct[va_idx]
    )
    oof_delta_lgbm[va_idx] = preds_lgbm

    # Consensus delta (LGBM + XGB + Ridge)
    preds_cons, sims_cons = predict_single_template_delta(
        fps_va, fps_ft, y_ft, phys_va, phys_ft,
        [delta_lgbm, delta_xgb, delta_ridge], oof_direct[va_idx]
    )
    oof_consensus[va_idx] = preds_cons

    r_dir  = rae(y_tr[va_idx], oof_direct[va_idx])
    r_lgbm = rae(y_tr[va_idx], oof_delta_lgbm[va_idx])
    r_cons = rae(y_tr[va_idx], oof_consensus[va_idx])
    print(f"  fold {fold+1}  direct={r_dir:.4f}  delta_lgbm={r_lgbm:.4f}  "
          f"consensus={r_cons:.4f}  avg_sim={sims_cons.mean():.3f}", flush=True)

m_dir   = full_metrics(y_tr, oof_direct,     cliff_pairs, "direct_lgbm")
m_dlgbm = full_metrics(y_tr, oof_delta_lgbm, cliff_pairs, "delta_lgbm_only")
m_cons  = full_metrics(y_tr, oof_consensus,  cliff_pairs, "consensus_delta")
print("\n" + pd.DataFrame([m_dir, m_dlgbm, m_cons],
                           index=["direct","delta_lgbm","consensus"]).round(4).to_string())



=== Scaffold 5-fold CV ===


  fold 1  direct=0.4982  delta_lgbm=0.3038  consensus=0.4625  avg_sim=0.394


  fold 2  direct=0.5759  delta_lgbm=0.3283  consensus=0.5174  avg_sim=0.391


  fold 3  direct=0.6021  delta_lgbm=0.3685  consensus=0.5596  avg_sim=0.387


  fold 4  direct=0.5665  delta_lgbm=0.3375  consensus=0.5313  avg_sim=0.388


  fold 5  direct=0.6033  delta_lgbm=0.3565  consensus=0.5577  avg_sim=0.390


  [direct_lgbm] RAE=0.5643 MAE=0.5134 R2=0.5991 r=0.7740 rho=0.7268 tau=0.5345
  [delta_lgbm_only] RAE=0.3361 MAE=0.3058 R2=0.8138 r=0.9035 rho=0.8719 tau=0.7157
  [consensus_delta] RAE=0.5212 MAE=0.4742 R2=0.6786 r=0.8287 rho=0.7753 tau=0.5840

               RAE     MAE      R2  Pearson  Spearman  Kendall
direct      0.5643  0.5134  0.5991   0.7740    0.7268   0.5345
delta_lgbm  0.3361  0.3058  0.8138   0.9035    0.8719   0.7157
consensus   0.5212  0.4742  0.6786   0.8287    0.7753   0.5840


In [9]:
# ---- Blend sweep: consensus_delta vs direct ----
best_alpha, best_rae_v = 0.0, full_metrics(y_tr, oof_direct)["RAE"]
for alpha in np.arange(0.0, 1.05, 0.1):
    blended = alpha*oof_consensus + (1-alpha)*oof_direct
    r = rae(y_tr[np.isfinite(blended)], blended[np.isfinite(blended)])
    print(f"  alpha={alpha:.1f}  RAE={r:.4f}")
    if r < best_rae_v:
        best_rae_v, best_alpha = r, alpha

oof = best_alpha*oof_consensus + (1-best_alpha)*oof_direct
m_blend = full_metrics(y_tr, oof, cliff_pairs, f"blend(a={best_alpha:.1f})")
print(f"\nBest blend alpha={best_alpha:.1f}  OOF RAE={best_rae_v:.4f}")


  alpha=0.0  RAE=0.5643
  alpha=0.1  RAE=0.5531
  alpha=0.2  RAE=0.5433
  alpha=0.3  RAE=0.5350
  alpha=0.4  RAE=0.5282
  alpha=0.5  RAE=0.5233
  alpha=0.6  RAE=0.5200
  alpha=0.7  RAE=0.5180
  alpha=0.8  RAE=0.5173
  alpha=0.9  RAE=0.5184
  alpha=1.0  RAE=0.5212
  [blend(a=0.8)] RAE=0.5173 MAE=0.4706 R2=0.6788 r=0.8292 rho=0.7787 tau=0.5872

Best blend alpha=0.8  OOF RAE=0.5173


In [10]:
# ---- Final test predictions ----
print("\nFitting final direct model on all train...", flush=True)
m_final = lgb.train(LGBM, lgb.Dataset(X_tr, label=y_tr), callbacks=[lgb.log_evaluation(-1)])
te_direct = m_final.predict(X_te)

phys_te = np.array([[p.get(k,0) or 0 for k in props]
                     for p in te["smiles"].map(compute_physchem)], dtype=np.float32)

print("Running consensus delta on test...", flush=True)
te_cons, te_sims = predict_single_template_delta(
    fps_te, fps_tr, y_tr, phys_te, phys_arr,
    [delta_lgbm, delta_xgb, delta_ridge], te_direct
)
print(f"Test: sim range [{te_sims.min():.3f}, {te_sims.max():.3f}]  "
      f"mean={te_sims.mean():.3f}")

te_preds = best_alpha*te_cons + (1-best_alpha)*te_direct
te_preds = np.clip(te_preds, y_tr.min()-0.5, y_tr.max()+0.5)

np.save(DATA_PROCESSED/"oof_consensus_delta_ml.npy", oof)
np.save(DATA_PROCESSED/"te_oof_consensus_delta_ml.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"99_consensus_delta_ml.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")
print(f"\n*** nb99 OOF RAE = {m_blend['RAE']:.4f} ***")



Fitting final direct model on all train...


Running consensus delta on test...


Test: sim range [0.323, 0.806]  mean=0.532
Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\99_consensus_delta_ml.csv
Test: min=3.17 med=4.89 max=5.73

*** nb99 OOF RAE = 0.5173 ***
